# Solutions · Chapter 00-03 · The kinds of learning, and the map of the work

Worked answers with reasoning. Two of these (E6 and E14) produce results that contradict what
most people expect, and those are the two worth reading most carefully.

Self-contained: run from the top with a fresh kernel.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(1)
n_days = 60
temp_c = rng.integers(10, 29, size=n_days)
is_rainy = rng.random(n_days) < 0.40
rentals = 1.5 * temp_c - 22 * is_rainy + rng.normal(0, 3, n_days)

season = pd.DataFrame({
    "day": np.arange(1, n_days + 1), "temp_c": temp_c,
    "weather": np.where(is_rainy, "rainy", "sunny"),
    "rentals": np.clip(np.round(rentals), 0, None).astype(int),
})
train, test = season.iloc[:40], season.iloc[40:]
rainy_flag = (season["weather"] == "rainy").to_numpy()
Z = StandardScaler().fit_transform(season[["temp_c", "rentals"]])
print("60 SYNTHETIC days ready;", int(rainy_flag.sum()), "of them rainy")

## E1 · The four families, by where the answers come from

- **Supervised:** the answers were recorded by someone or something, for most rows, and you
  learn to reproduce them on new rows.
- **Unsupervised:** there are no answers at all, and you look for structure in the inputs
  themselves.
- **Self-supervised:** you manufacture the answers by hiding part of each row and predicting it
  from the rest, so no human labelling is needed.
- **Reinforcement:** the answers only exist after you act, arriving as rewards, and your actions
  determine what you get to observe next.

**The trap:** defining them by algorithm ("k-means is unsupervised") rather than by data
situation. The same algorithm can appear in several settings; the family is a property of *your
problem*, which is why the flowchart in the chapter asks only about what data you have.

## E2 · Regression or classification?

**The test:** is 7 nearer to 8 than to 2? If distance between values means something, the target
is numeric and you are doing regression.

| Target | Which | Why |
|---|---|---|
| House price | **Regression** | 300,001 is nearly 300,000; being 5,000 off is meaningfully different from being 200,000 off |
| Postcode | **Classification** (or neither) | Digits without arithmetic - the next postcode is not the next-door house. Usually a category, often better used as a lookup to real features |
| Star rating out of 5 | **Either, and it matters** | The values are ordered (4 beats 3) but the gaps may not be equal. Called *ordinal*; treating it as regression assumes equal gaps, as classification throws away the order. Say which assumption you are making |
| Number of children | **Regression**, with care | A genuine count: 3 is between 2 and 4. But it cannot be negative or fractional, so a plain linear model can predict impossible values - see 05-05 |
| Blood type | **Classification** | A, B, AB, O have no order and no distance |

**The instinct being trained:** the star rating row. Most real targets are not cleanly one or
the other, and the professional move is to name the assumption you are making rather than to pick
silently.

## E3 · Why a clustering cannot be validated like a prediction

A prediction can be checked because the true answer exists and was withheld: you predict, you
uncover, you measure the gap. Every tool from chapter 00-01 - baseline, held-out set, MAE -
depends on there *being* a right answer.

A clustering has none. Nothing in the data says how many groups there should be or which rows
belong together, so there is no gap to measure and no baseline to beat. Internal scores such as
silhouette measure whether the groups are *tidy* (compact and separated), which is a statement
about geometry, not about truth: a tidy partition of structureless data scores well.

**Consequence:** every real check on a clustering has to come from outside the algorithm -
stability under resampling, agreement with a variable that was held back, or a domain expert
recognising the groups. Chapter 08-06 builds these.

## E4 · Naming a cluster from its composition

- Group 0: `6 / 30 =` **0.20** rainy
- Group 1: `14 / 30 =` **0.467** rainy
- Season overall: `20 / 60 =` **0.333** rainy

**Would you call group 1 "the rainy cluster"? No.**

It is *enriched* in rainy days - 0.467 against a base rate of 0.333, a lift of about 1.4 - which
is a real signal and worth reporting. But **53% of group 1 is sunny**, so anyone acting on the
name "the rainy cluster" would be wrong about the majority of its members. And the base rate is
what makes this judgement possible: without it, 0.467 looks like "mostly rainy" instead of
"slightly more rainy than average".

**The habit:** whenever a group is described by a label, ask for the label's share *inside* the
group and *in the population*. Almost every overclaimed segment in industry dies at that
question.

In [ ]:
print("group 0 share rainy:", round(6 / 30, 3))
print("group 1 share rainy:", round(14 / 30, 3))
print("season   share rainy:", round(20 / 60, 3), " -> lift for group 1:", round((14 / 30) / (20 / 60), 2))

## E5 · The cost of labels

- All 1,000: `1000 x 90 s = 90,000 s = 25 hours`.
- 200 labels: `200 x 90 s = 18,000 s = 5 hours`.
- **Saved: 20 hours** - four days of someone's attention.

That is why semi-supervised and active learning exist. Labels are the expensive ingredient, and
in specialist domains (a radiologist, a lawyer) the cost is far higher than 90 seconds.

**The risk that saving carries:** the 200 labelled examples now determine everything, and any
bias in *which* 200 got labelled propagates into the model with confidence. If the first 200
photos were the well-lit ones from the front of the warehouse, the model learns "well-lit and
from the front" and the unlabelled data cannot correct it.

**A second, sharper risk:** you still need labelled data to *check* the result. A held-out
labelled set is not optional, and it comes out of the same budget. Claims like "the same
accuracy with 200 labels" are only meaningful if someone labelled a test set too.

## E6 · Which columns you feed decides what you "discover"

In [ ]:
for columns in (["temp_c"], ["rentals"], ["temp_c", "rentals"]):
    Zc = StandardScaler().fit_transform(season[columns])
    labels = KMeans(n_clusters=2, n_init=10, random_state=0).fit(Zc).labels_
    # the labels are arbitrary (0/1), so take whichever assignment matches better
    agreement = max((labels == rainy_flag).mean(), (labels != rainy_flag).mean())
    print(f"clustered on {str(columns):<24} agreement with weather = {agreement:.3f}")

**Temperature alone: 0.533.** Barely better than tossing a coin, and rightly so - temperature was
generated independently of rain, so grouping by it says nothing about weather.

**Rentals alone: 0.783.** The best of the three. Rain is what pushes rentals down by 22, so the
rental count carries the weather signal more directly than anything else here.

**Both together: 0.633.** *Worse than rentals alone.* Adding a second column did not add
information about weather - it added a strong, irrelevant axis that the distance calculation now
has to share attention with, diluting the signal that was there.

**The lesson, which is the point of the exercise:** an unsupervised result is a function of the
columns you chose and how you scaled them. Change the columns and you get different "discoveries"
from the same days. Since there is no held-out score to catch you, nothing in the output warns
you that a column choice quietly changed your conclusion.

**Transferable version:** whenever someone shows you clusters, ask *which features went in, and
how were they scaled?* Those two answers determine the result more than the algorithm does.
Chapter 08-02 shows scaling doing exactly this.

## E7 · Are the k=4 groups stable?

If the groups reflect real structure, re-clustering a random 80% of the days should keep the same
days together. If they are an artefact of the partition, membership will shuffle.

The measure below is **pair co-membership**: for every pair of days, were they in the same group
in the full run, and are they still in the same group in the resample?

In [ ]:
full_labels = KMeans(n_clusters=4, n_init=10, random_state=0).fit(Z).labels_
together_full = full_labels[:, None] == full_labels[None, :]

boot = np.random.default_rng(0)
agreements, preserved = [], []
for _ in range(50):
    idx = boot.choice(n_days, 48, replace=False)                      # 80% of the days
    labels = KMeans(n_clusters=4, n_init=10, random_state=0).fit(Z[idx]).labels_
    together = labels[:, None] == labels[None, :]
    reference = together_full[np.ix_(idx, idx)]
    upper = np.triu_indices(len(idx), 1)
    agreements.append((together[upper] == reference[upper]).mean())
    preserved.append(together[upper][reference[upper]].mean())

print(f"pairs whose co-membership is unchanged : {np.mean(agreements):.3f}")
print(f"pairs together before, still together  : {np.mean(preserved):.3f}")

**92% of pairs keep the same relationship, but only 83% of the pairs that were grouped together
stay together.** Roughly one in six co-memberships dissolves when you drop a fifth of the days.

**Would you name these groups in a meeting?** Not without saying that. "Four groups, of which
about one in six memberships changes if we resample the data" is an honest sentence. "We
identified four day-types" is not, and it is the sentence that gets said.

Note also what the second number catches that the first misses. With four groups, most pairs are
in *different* groups, and "still in different groups" is easy to get right - so the headline 92%
is flattered by the easy cases. Looking specifically at pairs that were together is the harder
and more informative question. **Choosing the denominator honestly is a skill this course returns
to** in 06-05, where precision and recall are exactly this choice.

## E8 · "So the model understands the weather"

**Two reasons for care:**

1. **It inverted an arithmetic relationship we built ourselves.** We generated rentals as
   `1.5 x temperature - 22 x rain + noise`. Given rentals and weather, recovering temperature is
   algebra, not insight. The model found a relationship that was placed there by three lines of
   code.
2. **"Understands" implies knowing when not to answer.** Hand it a January day at -5 °C, a
   festival, or a day when the stand was closed, and it will produce a confident number anyway.
   A system that understood would hesitate. This one has no representation of its own scope.

**What the result actually establishes:** that temperature is largely recoverable from rentals
and weather *in this data*, to within 1.23 °C, over the range of days observed. That is a claim
about a correlation inside a range, not about comprehension.

**Why it matters beyond word-choice:** if you believe a model "understands", you will trust it
outside the range it was fitted on. That is where models fail hardest and most silently - and it
is the reason chapter 13-03 asks every model card to state an intended-use range explicitly.

## E9 · "The anomaly detector flags 5% and most flags are fine"

**A framing problem.** "Anomalous" and "fraudulent" are different concepts. An unsupervised
detector finds rows that are *unusual*; a very large legitimate order is unusual and not fraud,
while a competent fraudster deliberately looks ordinary. If the goal is fraud and labelled fraud
cases exist, this should be a supervised problem, and if only a few labels exist, semi-supervised
- not unsupervised.

**A data problem.** The 5% may be an artefact of the inputs: an unscaled column dominating the
distance, a field that is null for one system's transactions, a new merchant category, a
timestamp in a different timezone. Unusual-looking rows are frequently just differently-recorded
rows.

**An evaluation problem.** "Most flags are fine" describes only the flagged rows. Nobody has
looked at the 95% *not* flagged, so the misses are invisible. A detector could be catching 2% of
real fraud and this report would read the same. Chapter 06-05 names these two views precision and
recall, and the trap of only ever seeing one of them.

**What to look at first:** the 5%. Take twenty flagged rows and read them next to twenty
unflagged ones. If the flagged rows share a boring technical property - one merchant, one null
field, one currency - it is the data problem, which is cheap to fix. That inspection costs an hour
and routinely ends the investigation.

## E10 · A million unlabelled records, budget for 500 labels

> First I would define the target precisely, because 500 labels of a vague target are worthless.
> Then I would spend a few of the labels immediately on a **held-out test set**, because without
> one I cannot tell whether anything I do afterwards helps. For choosing the remaining labels I
> would use **active learning** rather than a random sample: label a small random batch, fit a
> model, then label the cases it is least certain about, which typically reaches a given accuracy
> with far fewer labels. Alongside that I would try **self-supervised or transfer learning** on
> the million unlabelled records to get a representation, then fit a small supervised model on top
> of it with my few labels. I would try the simple supervised model on a random sample first,
> because it is one afternoon of work and it sets the baseline everything else has to beat.

**What the question is testing:** whether you protect an evaluation budget before spending the
training budget. Most candidates spend all 500 on training and then cannot demonstrate anything.

## E11 · Online learning for bank fraud?

**For:** fraud patterns change within hours - a new technique appears and spreads - so a model
retrained monthly is describing last month's fraud. Online updating adapts as the attack evolves,
and the transaction volume easily supports continuous learning.

**Against:** an online model is a moving target that is hard to test, reproduce, audit and roll
back, and in fraud the *adversary can feed it inputs*. If an attacker can influence what the model
learns from, continuous updating becomes an attack surface: run cheap transactions of a certain
shape until the model treats that shape as normal. Labels also arrive late - a chargeback can take
weeks - so "online" updating would often be learning from labels that are not yet correct.

**Recommendation:** batch retraining on a short, frequent schedule, with fast-updating *rules and
lists* handling the hours-scale reaction, and an online component only if you can guarantee
label latency is short and the update path is protected against poisoning. The condition matters
more than the choice: adopt online learning only when you have monitoring good enough to notice
the model degrading within the same time window it can degrade in.

## E12 · The hospital's four assets

| Asset | Family | A question it could answer |
|---|---|---|
| 12 years of admissions with outcomes | **Supervised** | Predict length of stay (regression) or readmission (classification) |
| 400,000 unlabelled chest X-rays | **Self-supervised**, then transfer | Pre-train a representation that makes the 2,000 labelled images go much further |
| 2,000 X-rays labelled by a radiologist | **Supervised**, ideally on top of the above; **active learning** to choose the next 2,000 | Detect a specific finding |
| A proposal to adjust triage and see what happens | **Reinforcement learning / an experiment** | What triage policy improves outcomes |

**The one that should probably not be attempted: the triage policy by reinforcement learning.**
It is the only one where the system's actions change what happens to patients, and where learning
requires *exploring* - trying actions to see what they do. Exploration on people is not a
modelling decision, it is a clinical trial, with ethics approval, consent and oversight. The
correct route is a carefully designed prospective study, not an agent.

The 12 years of admissions carry a quieter version of the same problem: they record outcomes
under the *existing* triage policy, so a model fitted to them learns what happened under that
policy and cannot tell you what a different one would do. That is exactly the confusion chapter
00-04 is about.

## E13 · Explaining it to Maria

> The computer did not find three types. It was told to make three piles and it made three piles.
> Ask it for two and it makes two, ask for five and it makes five, and every version looks tidy.
> The piles are real in the sense that the days in each are similar - but "there are three kinds
> of customer" is our sentence, not the computer's finding. Before believing it, we should check
> whether the same piles appear when we leave some days out.

(76 words.)

**The instinct:** the difference between *"the algorithm produced"* and *"we discovered"* is not
pedantry. It is the difference between a hypothesis and a claim, and it is the sentence that
decides whether a business reorganises itself around a number someone typed.

## E14 · A semi-supervised experiment, honestly reported

Hide the weather label for most training days, train on the few that remain, then use the model's
own confident guesses as labels and retrain. This is **pseudo-labelling**, the simplest
semi-supervised method.

To make it a real test we use only `rentals` as the feature, so the two classes genuinely overlap
- and we repeat the whole thing 20 times with different randomly chosen labelled days, because a
single run of a 6-label experiment tells you nothing.

In [ ]:
FEATURES = ["rentals"]
before, after = [], []

for seed in range(20):
    r = np.random.default_rng(seed)
    labelled = np.zeros(40, dtype=bool)
    labelled[r.choice(40, 6, replace=False)] = True
    if len(set(train["weather"][labelled])) < 2:      # need both classes to fit
        continue

    first = LogisticRegression(max_iter=1000).fit(train[FEATURES][labelled], train["weather"][labelled])
    before.append(accuracy_score(test["weather"], first.predict(test[FEATURES])))

    hidden = train[FEATURES][~labelled]
    confident = first.predict_proba(hidden).max(axis=1) > 0.90        # trust only the sure ones
    X = pd.concat([train[FEATURES][labelled], hidden[confident]])
    y = np.concatenate([train["weather"][labelled], first.predict(hidden)[confident]])
    after.append(accuracy_score(test["weather"], LogisticRegression(max_iter=1000).fit(X, y).predict(test[FEATURES])))

before, after = np.array(before), np.array(after)
print(f"{len(before)} runs, 6 labelled days each")
print(f"  6 labels only    : mean {before.mean():.3f}  (min {before.min():.3f}, max {before.max():.3f})")
print(f"  + pseudo-labels  : mean {after.mean():.3f}  (min {after.min():.3f}, max {after.max():.3f})")
print(f"  helped {(after > before).sum()}, hurt {(after < before).sum()}, unchanged {(after == before).sum()}")
print(f"  all 40 labels    : "
      f"{accuracy_score(test['weather'], LogisticRegression(max_iter=1000).fit(train[FEATURES], train['weather']).predict(test[FEATURES])):.3f}")

**It did not help.** Mean accuracy went from **0.868 to 0.863**; it improved 1 run, hurt 3, and
changed nothing in 16.

Look also at the last line: training on **all 40 labels** scores 0.850 - *lower* than the average
6-label run. That is not a finding about label efficiency. It is a 20-day test set being far too
small to distinguish these models, and it is the more important lesson of the two: differences of
this size, measured this way, are noise. Chapter 03-03 gives you the tool to say how much noise,
and 07-03 the machinery to compare models without fooling yourself.

**When does pseudo-labelling actively make things worse?** When the first model is *confidently
wrong*. Its mistakes are converted into training labels, the retrained model becomes more certain
about them, and the error is now baked in and self-reinforcing - with no signal that anything
happened, because the pseudo-labels agree with the model by construction. The danger is highest
exactly where the method is most tempting: very few labels, so the first model is unreliable, and
a high confidence threshold that feels like a safeguard but only selects the cases the model is
most sure about - including the ones it is most sure and wrong about.

**When it does work:** when unlabelled data is plentiful, the classes are genuinely well
separated, and the initial model is decent rather than desperate. Chapter 12-01 covers the
conditions, and active learning - letting the model *ask* which examples to label - which is
usually the better use of a small budget.

---

## Where to go next

Back to the chapter for the mastery check and flashcards, then **00-04 · Prediction, explanation,
and cause**, the last orientation chapter.